# 01 — Synthetic Dataset Generation

## Purpose
This notebook generates the 225,000-row synthetic telecom churn dataset used throughout
the project. Because this dataset is fully synthetic:

- **Reproducibility** — anyone can regenerate the exact data from a fixed random seed.
- **No privacy risk** — no real customer records are involved.
- **Controlled realism** — the churn logic mirrors known retention dynamics without
  overfitting to any single carrier's quirks.

## Design Principles
1. **Scale** — 225k rows approximate a meaningful slice of a large carrier's subscriber base.
2. **Leakage-aware** — columns that directly encode the target (`churn_score`, `cltv`,
   `billing_risk_score`) are generated but **excluded from ML features** downstream.
3. **Transparent data contract** — a JSON metadata file documents every column's role
   and any gotchas (nullable fields, derived columns, etc.).

## Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ── Global Config ─────────────────────────────────────────────────────────────
N_CUSTOMERS = 225_000
SEED        = 42
VERSION     = "1.2.0"

# Contract distribution: 55% month-to-month, 25% one-year, 20% two-year
P_CONTRACT     = [0.55, 0.25, 0.20]
CONTRACT_TYPES = ["month_to_month", "one_year", "two_year"]

# Service segment distribution
P_SEGMENT = [0.30, 0.40, 0.15, 0.15]  # internet_only, internet_plus_other, cable_only, mobile
SEGMENTS  = ["internet_only", "internet_plus_other", "cable_only", "mobile"]

# Churn sigmoid parameters (calibrated to produce ~22-25% base churn rate)
CHURN_SIGMOID_THRESHOLD = 45
CHURN_LOGISTIC_SCALE    = 12

# Pricing anchors
BASE_MONTHLY_FIBER = 70.0
BASE_MONTHLY_DSL   = 45.0

np.random.seed(SEED)

## Step 1 — Demographics, Contracts & Billing

Basic demographic fields are drawn from population-appropriate distributions.
Contract type is the single strongest predictor of churn in most real-world telco
datasets, so we assign it early and let it propagate into the churn score later.

In [ ]:
# Demographics
customer_id      = [f"{i:07d}-TEL" for i in range(N_CUSTOMERS)]
gender           = np.random.choice(["male", "female"], N_CUSTOMERS)
senior_citizen   = np.random.choice([0, 1], N_CUSTOMERS, p=[0.84, 0.16])
partner          = np.random.choice(["yes", "no"], N_CUSTOMERS)
dependents       = np.random.choice(["yes", "no"], N_CUSTOMERS, p=[0.3, 0.7])
tenure_months    = np.random.randint(1, 73, N_CUSTOMERS)

# Contract & billing
contract         = np.random.choice(CONTRACT_TYPES, N_CUSTOMERS, p=P_CONTRACT)
paperless_billing = np.random.choice(["yes", "no"], N_CUSTOMERS, p=[0.6, 0.4])
payment_method   = np.random.choice(
    ["electronic_check", "mailed_check", "bank_transfer_auto", "credit_card_auto"],
    N_CUSTOMERS, p=[0.34, 0.23, 0.22, 0.21]
)
is_autopay = np.where(np.isin(payment_method, ["bank_transfer_auto", "credit_card_auto"]), 1, 0)

## Step 2 — Service Segments & Add-Ons

**Key business logic:** cable-only customers have no internet service; every other
segment requires internet (fiber or DSL). This constraint is enforced explicitly to
prevent nonsensical combinations in the dataset.

In [ ]:
service_segment = np.random.choice(SEGMENTS, N_CUSTOMERS, p=P_SEGMENT)

internet_service = np.array([
    "no" if s == "cable_only" else np.random.choice(["fiber", "dsl"], p=[0.7, 0.3])
    for s in service_segment
])

phone_service  = np.where(np.isin(service_segment, ["internet_plus_other", "mobile"]), "yes", "no")
multiple_lines = np.where(
    phone_service == "yes",
    np.random.choice(["no", "yes", "no_phone_service"], N_CUSTOMERS, p=[0.5, 0.4, 0.1]),
    "no_phone_service",
)

# Add-ons (internet customers only)
def inet_feature(p_yes):
    return np.where(
        internet_service != "no",
        np.random.choice(["no", "yes"], N_CUSTOMERS, p=[1 - p_yes, p_yes]),
        "no_internet_service",
    )

online_security  = inet_feature(0.30)
online_backup    = inet_feature(0.40)
device_protection = inet_feature(0.40)
tech_support     = inet_feature(0.30)
streaming_tv     = inet_feature(0.50)
streaming_movies = inet_feature(0.50)

## Step 3 — Promo, Sales Channel & Early Friction Events

Early friction is one of the most operationally actionable signals in retention
analytics. We simulate two friction types — billing issues and tech issues — within
the first 60 days of service. Resolution status is stored as a **nullable integer**:
`NA` means no issue occurred; `0` = unresolved; `1` = resolved.

This nullable design is intentional: it prevents downstream models from treating
"no call" and "unresolved call" as the same class.

In [ ]:
sales_channel = np.random.choice(
    ["online", "store", "agent_call_center", "third_party_retailer"],
    N_CUSTOMERS, p=[0.4, 0.25, 0.25, 0.1],
)

is_on_promo       = np.random.choice([1, 0], N_CUSTOMERS, p=[0.35, 0.65])
promo_discount_pct = np.where(
    is_on_promo == 1,
    np.random.choice([0.1, 0.2, 0.3, 0.5], N_CUSTOMERS),
    0.0,
)

first_60d_billing_call = np.random.choice([1, 0], N_CUSTOMERS, p=[0.15, 0.85])
billing_call_resolved  = np.where(
    first_60d_billing_call == 1,
    np.random.choice([1, 0], N_CUSTOMERS, p=[0.7, 0.3]),
    np.nan,
)

first_60d_tech_call = np.random.choice([1, 0], N_CUSTOMERS, p=[0.12, 0.88])
tech_issue_resolved = np.where(
    first_60d_tech_call == 1,
    np.random.choice([1, 0], N_CUSTOMERS, p=[0.6, 0.4]),
    np.nan,
)

## Step 4 — Pricing & Total Charges

`monthly_charges` reflects the full-rate price. `monthly_charges_billed` applies
any active promo discount. `total_charges` uses a **piecewise rule**: promo rate for
the first 12 months, then full rate thereafter — mimicking how most promotional
contracts actually work.

In [ ]:
monthly_charges  = np.where(internet_service == "fiber", BASE_MONTHLY_FIBER, BASE_MONTHLY_DSL)
monthly_charges  = np.where(internet_service == "no", 30.0, monthly_charges)
monthly_charges += (streaming_tv    == "yes").astype(int) * 15.0
monthly_charges += (streaming_movies == "yes").astype(int) * 15.0
monthly_charges += (phone_service   == "yes").astype(int) * 10.0
monthly_charges += np.random.normal(0, 3, N_CUSTOMERS)
monthly_charges  = np.round(np.clip(monthly_charges, 20.0, 150.0), 2)

monthly_charges_billed = np.where(
    is_on_promo == 1,
    monthly_charges * (1 - promo_discount_pct),
    monthly_charges,
)

total_charges = []
for i in range(N_CUSTOMERS):
    t, full = tenure_months[i], monthly_charges[i]
    if is_on_promo[i] == 1:
        p = monthly_charges_billed[i]
        total_charges.append(round(p * min(t, 12) + full * max(0, t - 12), 2))
    else:
        total_charges.append(round(full * t, 2))

## Step 5 — Churn Probability Model

The synthetic churn target is generated via a **logistic sigmoid** applied to a
weighted `churn_score`. The score accumulates points for known risk factors:

| Risk Factor                        | Points |
|------------------------------------|--------|
| Month-to-month contract            | +35    |
| On promotional pricing             | +10    |
| Fiber internet                     | +15    |
| Not on autopay                     | +12    |
| Billing call in first 60 days      | +10    |
| **Unresolved** billing issue       | +25    |
| **Unresolved** tech issue          | +20    |
| Tenure < 12 months                 | +15    |
| Senior citizen                     | +5     |
| Random noise (σ=10)                | ±      |

The sigmoid threshold is calibrated so that the base churn rate lands near **22–25%**,
consistent with typical mid-size telco attrition.

> **Important:** `churn_score` is intentionally excluded from all ML features downstream.
> It is the synthetic driver of the target, not an observable business signal.

In [ ]:
churn_score  = np.zeros(N_CUSTOMERS)
churn_score += (contract == "month_to_month") * 35
churn_score += (is_on_promo == 1) * 10
churn_score += (internet_service == "fiber") * 15
churn_score += (is_autopay == 0) * 12
churn_score += (first_60d_billing_call == 1) * 10
churn_score += (billing_call_resolved == 0) * 25
churn_score += (tech_issue_resolved == 0) * 20
churn_score += (tenure_months < 12) * 15
churn_score += (senior_citizen == 1) * 5
churn_score += np.random.normal(0, 10, N_CUSTOMERS)
churn_score  = np.clip(churn_score, 0, 100)

sigmoid    = lambda x: 1 / (1 + np.exp(-(x - CHURN_SIGMOID_THRESHOLD) / CHURN_LOGISTIC_SCALE))
churn_prob = sigmoid(churn_score)
churn      = np.random.binomial(1, churn_prob)

print(f"Overall churn rate: {churn.mean():.1%}")
print(f"Total churners: {churn.sum():,}")

## Step 6 — Derived Scores & Final Assembly

Three columns are computed for dashboard / financial analysis use only.
They are **never used as ML features**:

- `billing_risk_score` — deterministic segmentation score from contract + payment type
- `cltv` — 24-month contribution margin estimate per customer
- `churn_score` — the synthetic target driver (see above)

In [ ]:
estimated_margin_pct = np.round(np.random.uniform(0.15, 0.35, N_CUSTOMERS), 4)
cltv                 = np.round(monthly_charges * 24 * estimated_margin_pct, 2)
billing_risk_score   = ((is_autopay == 0) * 40 + (paperless_billing == "no") * 20
                        + (contract == "month_to_month") * 40).astype(int)

df = pd.DataFrame({
    "customer_id": customer_id, "gender": gender, "senior_citizen": senior_citizen,
    "partner": partner, "dependents": dependents, "tenure_months": tenure_months,
    "phone_service": phone_service, "multiple_lines": multiple_lines,
    "internet_service": internet_service, "online_security": online_security,
    "online_backup": online_backup, "device_protection": device_protection,
    "tech_support": tech_support, "streaming_tv": streaming_tv,
    "streaming_movies": streaming_movies, "service_segment": service_segment,
    "contract_type": contract, "paperless_billing": paperless_billing,
    "payment_method": payment_method,
    "monthly_charges": np.round(monthly_charges, 2),
    "monthly_charges_billed": np.round(monthly_charges_billed, 2),
    "total_charges": total_charges, "is_on_promo": is_on_promo,
    "promo_discount_pct": promo_discount_pct, "sales_channel": sales_channel,
    "first_60d_billing_call": first_60d_billing_call,
    "billing_call_resolved": billing_call_resolved,
    "first_60d_tech_call": first_60d_tech_call,
    "tech_issue_resolved": tech_issue_resolved,
    "is_autopay": is_autopay,
    "estimated_margin_pct": estimated_margin_pct,
    "cltv": cltv, "billing_risk_score": billing_risk_score,
    "churn_score": np.round(churn_score, 2), "churn": churn.astype("int64"),
})

df["billing_call_resolved"] = df["billing_call_resolved"].astype("Int64")
df["tech_issue_resolved"]   = df["tech_issue_resolved"].astype("Int64")

out_path = "../data/telco_churn_225k_v120.csv"
df.to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(df):,} rows x {len(df.columns)} columns)")
print(df["churn"].value_counts().rename("count"))